In [ ]:
import pyspark
from pyspark.sql import SparkSession

In [ ]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('taxi_yellow_green') \
    .getOrCreate()

In [ ]:
# Spark automatically combines into a single DataFrame
df = spark.read.parquet("taxi_trip/data/taxi_2020_clean/")
df.printSchema()
df.show(5)

In [ ]:
df.createOrReplaceTempView("trips_data")

In [ ]:
# Number of trips by service type
spark.sql("""
SELECT service_type, COUNT(*) AS total_trips
FROM trips_data
GROUP BY service_type
ORDER BY total_trips DESC
""").show()

In [ ]:
# Average distance by service type
spark.sql("""
SELECT service_type, ROUND(AVG(trip_distance),2) AS avg_distance
FROM trips_data
GROUP BY service_type
""").show()

In [ ]:
# Total revenue by month and service type
spark.sql("""
SELECT 
    year,
    month,
    service_type,
    ROUND(SUM(total_amount),2) AS revenue
FROM trips_data
GROUP BY year, month, service_type
ORDER BY year, month, service_type
""").show()


In [ ]:
# Average trip time by service type
spark.sql("""
SELECT 
    service_type,
    AVG(UNIX_TIMESTAMP(dropoff_datetime) - UNIX_TIMESTAMP(pickup_datetime)) / 60 AS avg_trip_minutes
FROM trips_data
GROUP BY service_type
""").show()